# Deep Learning 2026 – Assignment 1, Part 4: MLOps
The learning goal of this part of the assignment is to get hands-on experience with using an MLOps
framework in practice.
Below, you are asked to use an MLOps platform such as Weights and Biases. Tutorial and example
notebook using W&B for tasks similar to the below are available on Absalon (week 2 module). You are
free to use other similar frameworks as long as your choice supports the requested tasks.
Your tasks are to:

1. Consider the famous MNIST hand-written digit classification task. Write a program (use of AI
coding agent is encouraged) that follows a standard structure for deep learning including data
download, model specification, mini-batch generator that evaluates both training and test loss and
a training loop. Test you code on a simple network with one hidden layer with 42 hidden units.
The solution has to be handed in MNISTfeedforwardAssignment.ipynb


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

import wandb

In [24]:
# Data download
transform = transforms.ToTensor()

train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)
test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

In [25]:
# Mini bacthes
batch_size = 64
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

In [26]:
# Model specification
class FeedForwardNN(nn.Module):
    def __init__(self, hidden_units=42):
        super().__init__()
        self.hidden = nn.Linear(28 * 28, hidden_units)
        self.output = nn.Linear(hidden_units, 10)

    def forward(self, x):
        x = x.view(-1, 28 * 28)
        x = torch.relu(self.hidden(x))
        x = self.output(x)
        return x


In [27]:
# Evaluate training and test loss
def evaluate_loss(model, data_loader, criterion):
    model.eval()
    total_loss = 0.0
    total_samples = 0

    with torch.no_grad():
        for images, labels in data_loader:
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * images.size(0)
            total_samples += images.size(0)

    return total_loss / total_samples

In [28]:
# Training function for W&B sweep
def train(config=None):
    with wandb.init(config=config):
        config = wandb.config

        model = FeedForwardNN(hidden_units=config.hidden_units)
        criterion = nn.CrossEntropyLoss()

        optimizer = optim.Adam(
            model.parameters(),
            lr=config.learning_rate
        )

        train_loader = DataLoader(
            train_dataset,
            batch_size=config.batch_size,
            shuffle=True
        )

        test_loader = DataLoader(
            test_dataset,
            batch_size=config.batch_size,
            shuffle=False
        )

        for epoch in range(config.epochs):
            model.train()

            for images, labels in train_loader:
                optimizer.zero_grad()
                outputs = model(images)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()

            train_loss = evaluate_loss(model, train_loader, criterion)
            test_loss = evaluate_loss(model, test_loader, criterion)

            wandb.log({
                "epoch": epoch + 1,
                "train_loss": train_loss,
                "test_loss": test_loss
            })

2. Modify the notebook to log the training progress in the MLOps platform, and show the resulting
plots in your report. Describe the modifications you made to use the MLOps platform and include
code excerpts of the modifications in your report (should only be a few lines of code). If it goes too
slow to run on the entire MNIST dataset, you are welcome to reduce the training set size.


3. Implement a hyperparameter sweep using an MLOps platform, e.g., a W&B sweep. Sweep over the
parameters you find most important (but keep the sweep of a size compatible with the compute
resources you have - the size of the sweep will not affect the grading). Visualize the results using,
e.g., a parallel coordinate plot in W&B.


In [29]:
sweep_config = {
    "method": "random",
    "metric": {
        "name": "test_loss",
        "goal": "minimize"
    },
    "parameters": {
        "learning_rate": {
            "values": [0.0001, 0.001, 0.01]
        },
        "batch_size": {
            "values": [32, 64, 128]
        },
        "hidden_units": {
            "values": [32, 42, 64]
        },
        "epochs": {
            "value": 10
        }
    }
}

In [30]:
sweep_id = wandb.sweep(
    sweep_config,
    project="mnist-feedforward"
)

wandb.agent(
    sweep_id,
    function=train,
    count=8
)

Create sweep with ID: uukuk7in
Sweep URL: https://forge.coreweave.com/wandb/lucashelmoe-k-benhavns-universitet/mnist-feedforward/sweeps/uukuk7in


wandb: Agent Starting Run: 06160izu with config:
wandb: 	batch_size: 64
wandb: 	epochs: 10
wandb: 	hidden_units: 64
wandb: 	learning_rate: 0.001
wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from C:\Users\Bruger\_netrc.


epoch,▁▂▃▃▄▅▆▆▇█
test_loss,█▅▄▃▂▂▂▁▁▁
train_loss,█▆▄▃▃▂▂▂▁▁
epoch,10
test_loss,0.09037
train_loss,0.04293


wandb: Agent Starting Run: 9a4qdot5 with config:
wandb: 	batch_size: 32
wandb: 	epochs: 10
wandb: 	hidden_units: 64
wandb: 	learning_rate: 0.01
wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from C:\Users\Bruger\_netrc.


epoch,▁▂▃▃▄▅▆▆▇█
test_loss,▂▁▂▃▃▅▅▅▄█
train_loss,█▅▆▄▅▅▄▃▁▅
epoch,10
test_loss,0.24923
train_loss,0.11479


wandb: Agent Starting Run: y5mhr6aj with config:
wandb: 	batch_size: 128
wandb: 	epochs: 10
wandb: 	hidden_units: 42
wandb: 	learning_rate: 0.0001
wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from C:\Users\Bruger\_netrc.


... (7 outputs left out)

wandb: Agent Starting Run: p82doue9 with config:
wandb: 	batch_size: 32
wandb: 	epochs: 10
wandb: 	hidden_units: 42
wandb: 	learning_rate: 0.0001
wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from C:\Users\Bruger\_netrc.


epoch,▁▂▃▃▄▅▆▆▇█
test_loss,█▅▄▃▃▂▂▂▁▁
train_loss,█▅▄▃▃▂▂▂▁▁
epoch,10
test_loss,0.21186
train_loss,0.20933


wandb: Agent Starting Run: nga5lmsi with config:
wandb: 	batch_size: 128
wandb: 	epochs: 10
wandb: 	hidden_units: 32
wandb: 	learning_rate: 0.0001
wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from C:\Users\Bruger\_netrc.


epoch,▁▂▃▃▄▅▆▆▇█
test_loss,█▄▃▂▂▁▁▁▁▁
train_loss,█▄▃▂▂▂▁▁▁▁
epoch,10
test_loss,0.28086
train_loss,0.2879


4. Describe the experimental setup and your results in the report. You must describe what you did,
present the results, discuss the results, and draw very careful preliminary conclusions. Discuss your
reasoning for the specific choice of parameters you considered. It is fine to lean on practices you
noticed in the literature.